# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdrayan001/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

#### Finding 1 — Content age differs between growing and declining pages

The paper reports that growing pages are younger on average than declining pages: about 185 days versus 228 days. The paper therefore presents content age as an important observed difference between the two groups.

**My methodology question:** How exactly is the growing/declining label defined, and is the label based on a future performance window that is separate from the page-age measurement? I would also want to know whether client mix and other page characteristics were controlled before treating age as an important signal.

I would describe this as an observed association, not proof that older content causes decline.

#### Finding 2 — CTR decreases as average position gets worse

The paper reports a strong difference in weighted CTR across position tiers, with the highest CTR in the top positions and much lower CTR for pages deeper in search results.

**My methodology question:** Are the position buckets and CTR calculated from the same observation window, and how is CTR weighted across pages with very different impression volumes? A cross-sectional relationship between position and CTR does not by itself show that improving position will cause a specific CTR increase.

For my own model, this finding supports checking CTR and average position as ranking signals, but it should not be treated as causal evidence.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Paper finding checks used in this section.
# These are reported observations, not causal claims.

paper_checks = {
    "finding_1": "Growing pages were reported as younger than declining pages.",
    "finding_2": "Weighted CTR was reported to decrease across worse position tiers.",
    "methodology_rule": "Both findings are treated as observed associations, not causal proof."
}

for key, value in paper_checks.items():
    print(f"{key}: {value}")

assert len(paper_checks) == 3
print("\nPaper-finding audit check passed.")

finding_1: Growing pages were reported as younger than declining pages.
finding_2: Weighted CTR was reported to decrease across worse position tiers.
methodology_rule: Both findings are treated as observed associations, not causal proof.

Paper-finding audit check passed.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5, the final model was evaluated with a client-grouped holdout. For this audit, I will compare that honest grouped result with a simpler random 75/25 split.

The random split is the "before" reference because pages from the same client can appear in both train and test sets. The grouped split is the "after" result because entire clients are held out from training.

I will use the same Random Forest model, the same feature set, the same decline proxy, and Precision@50 in both comparisons.

The purpose is not to make the score look better. It is to check how much the validation design changes the measured result.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------
# Load the starter dataset
# ---------------------------------------------------------

from pathlib import Path

data_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv")
]

data_path = next(
    (p for p in data_paths if p.exists()),
    None
)

assert data_path is not None, "Dataset file not found."

df_audit = pd.read_csv(data_path)

print("Loaded dataset:", data_path)
print("Dataset shape:", df_audit.shape)

df_audit["declining_proxy"] = (
    df_audit["trend_direction"] == "down"
).astype(int)

# Same Week-5 feature set
audit_features = [
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

# avg_position = 0 means no position data
df_audit["avg_position_missing"] = (
    df_audit["avg_position"] == 0
).astype(int)

df_audit.loc[
    df_audit["avg_position"] == 0,
    "avg_position"
] = np.nan

audit_features = audit_features + ["avg_position_missing"]

X = df_audit[audit_features]
y = df_audit["declining_proxy"]


# ---------------------------------------------------------
# BEFORE: ordinary random 75/25 split
# ---------------------------------------------------------

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )
)

random_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

random_model.fit(X_train_random, y_train_random)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]


def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(scores)[::-1]
    top_k = order[:k]

    return y_true[top_k].mean()


random_p50 = precision_at_k(
    y_test_random,
    random_scores,
    50
)


# ---------------------------------------------------------
# AFTER: client-grouped 75/25 split
# ---------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y=y,
        groups=df_audit["client_id"]
    )
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

grouped_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_scores = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_p50 = precision_at_k(
    y_test_grouped,
    grouped_scores,
    50
)


# ---------------------------------------------------------
# BEFORE vs AFTER comparison
# ---------------------------------------------------------

print("Validation comparison")
print("---------------------")
print(f"Random split Precision@50 : {random_p50:.3f}")
print(f"Grouped split Precision@50: {grouped_p50:.3f}")
print(f"Grouped test base rate    : {y_test_grouped.mean():.3f}")

print("\nRandom split test rows :", len(y_test_random))
print("Grouped split test rows:", len(y_test_grouped))

print("\nGrouped client overlap check:")

train_clients = set(
    df_audit.iloc[train_idx]["client_id"]
)

test_clients = set(
    df_audit.iloc[test_idx]["client_id"]
)

overlap = train_clients & test_clients

print("Train clients:", len(train_clients))
print("Test clients :", len(test_clients))
print("Overlap      :", len(overlap))

assert len(overlap) == 0

print("\nGrouped validation check passed.")

Loaded dataset: ..\..\data\raw\content_refresh_anonymized.csv
Dataset shape: (30000, 44)
Validation comparison
---------------------
Random split Precision@50 : 0.920
Grouped split Precision@50: 0.720
Grouped test base rate    : 0.517

Random split test rows : 7500
Grouped split test rows: 7115

Grouped client overlap check:
Train clients: 24
Test clients : 8
Overlap      : 0

Grouped validation check passed.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I will audit the final Week-5 feature set for direct or indirect leakage.

The target proxy is `trend_direction == "down"`, so `trend_direction` and `trend_pct` must not be features.

I will also exclude identifiers, product-derived flags, label-like fields, and other fields that could encode the target or a downstream decision.

The final model features are limited to observable search, engagement, and content-age signals:

- `impressions_90d`
- `sessions_90d`
- `ctr`
- `avg_position`
- `content_age_days`
- `days_since_last_update`
- `avg_position_missing`

The audit will check that none of the forbidden fields appear in the final feature set.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Leakage audit for the final Week-5 feature set
# ---------------------------------------------------------

final_features = [
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "avg_position_missing"
]

# Fields that must never be used as model features
forbidden_fields = [
    # Target / target-derived fields
    "trend_direction",
    "trend_pct",
    "declining_proxy",

    # Identifiers
    "content_id",
    "client_id",

    # Product / rule-derived flags
    "needs_ctr_fix",
    "is_quick_win",
    "needs_engagement_fix",
    "ai_opportunity",
    "is_underperformer",
    "is_initial_refresh_candidate",
    "health_score"
]

leaked_features = [
    field for field in forbidden_fields
    if field in final_features
]

print("Final model features:")
for feature in final_features:
    print(" -", feature)

print("\nForbidden fields found in final features:")
print(leaked_features)

assert leaked_features == [], (
    f"Potential leakage detected: {leaked_features}"
)

# Check that the target itself is not in the feature matrix
assert "declining_proxy" not in final_features

# Check that trend fields are excluded
assert "trend_direction" not in final_features
assert "trend_pct" not in final_features

print("\nLeakage audit passed.")
print("No target, target-derived, identifier, or product-derived fields are used.")

Final model features:
 - impressions_90d
 - sessions_90d
 - ctr
 - avg_position
 - content_age_days
 - days_since_last_update
 - avg_position_missing

Forbidden fields found in final features:
[]

Leakage audit passed.
No target, target-derived, identifier, or product-derived fields are used.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim:** The Random Forest model improves the identification of declining pages and can help the content team prioritize pages for review.

**Audited claim:** On this starter dataset, the Random Forest ranked pages with the current decline proxy more effectively than the Week-4 baseline on the held-out client-grouped test set, achieving 0.720 Precision@50 versus 0.620 for the baseline. However, the random-split result was much higher at 0.920, showing that validation design materially affects the measured result. These results are directional and decision-support evidence, not causal proof or a guarantee of future traffic recovery.

The model can therefore be used as a decision-support ranking for human review, with the client-grouped result treated as the more conservative estimate.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Final claim audit
# ---------------------------------------------------------

baseline_p50 = 0.620
grouped_p50 = grouped_p50
random_p50 = random_p50

print("Claim audit")
print("-----------")
print(f"Week-4 baseline Precision@50 : {baseline_p50:.3f}")
print(f"Random-split Precision@50    : {random_p50:.3f}")
print(f"Grouped Precision@50         : {grouped_p50:.3f}")

print("\nSupported claim:")
print(
    "The Random Forest ranked pages with the current decline proxy "
    "better than the Week-4 baseline on the held-out client-grouped test set."
)

print("\nImportant limitation:")
print(
    "The random split produced a much higher score than the grouped split, "
    "so the random-split result is likely optimistic for generalization to unseen clients."
)

print("\nClaim language check:")
print("Observed: PASS")
print("Measured: PASS")
print("Directional: PASS")
print("Decision-support: PASS")
print("Causal claim: NOT MADE")
print("Future performance guarantee: NOT MADE")

assert grouped_p50 > baseline_p50
assert len(train_clients & test_clients) == 0

print("\nFinal claim audit passed.")

Claim audit
-----------
Week-4 baseline Precision@50 : 0.620
Random-split Precision@50    : 0.920
Grouped Precision@50         : 0.720

Supported claim:
The Random Forest ranked pages with the current decline proxy better than the Week-4 baseline on the held-out client-grouped test set.

Important limitation:
The random split produced a much higher score than the grouped split, so the random-split result is likely optimistic for generalization to unseen clients.

Claim language check:
Observed: PASS
Measured: PASS
Directional: PASS
Decision-support: PASS
Causal claim: NOT MADE
Future performance guarantee: NOT MADE

Final claim audit passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.